BLOCK 1: Imports


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

print("Modules imported successfully.")

Modules imported successfully.


BLOCK 2: Data Loading & Initial Cleaning
Read files, forcing account_id to string to preserve leading zeros

In [2]:
df_lines = pd.read_csv('order_lines.csv', dtype={'account_id': str, 'zip_code_prefix': str})
df_payments = pd.read_csv('order_payments.csv')

# Convert date columns to datetime
df_lines['order_date'] = pd.to_datetime(df_lines['order_date'])

# Filter out canceled or unavailable orders
valid_statuses = ['delivered', 'shipped', 'invoiced', 'processing', 'created', 'approved']
df_lines = df_lines[df_lines['order_status'].isin(valid_statuses)]

print(f"Loaded {len(df_lines)} valid order lines.")

Loaded 102458 valid order lines.


BLOCK 3: Time Windows & Audience Filtering

In [3]:
# 1. Define Dates
cutoff_date = pd.to_datetime('2018-06-30')
max_date = pd.to_datetime('2018-08-31')

# 2. Split Data: Feature Window (Prior Data) vs. Target Window (Last 2 Months)
feature_data = df_lines[df_lines['order_date'] <= cutoff_date].copy()
target_data = df_lines[(df_lines['order_date'] > cutoff_date) & (df_lines['order_date'] <= max_date)].copy()

# 3. Filter for our target audience (Accounts with 10+ distinct orders in the feature window)
account_order_counts = feature_data.groupby('account_id')['order_id'].nunique()
established_accounts = account_order_counts[account_order_counts >= 10].index

feature_data = feature_data[feature_data['account_id'].isin(established_accounts)]
print(f"Found {len(established_accounts)} established accounts (10+ orders) in the feature window.")

Found 2149 established accounts (10+ orders) in the feature window.


BLOCK 4: Base Features & Payment Features

In [4]:
# Calculate freight burden per line
feature_data['freight_ratio'] = feature_data['freight_value'] / (feature_data['price'] + 0.01)

# Group core features by account
base_features = feature_data.groupby('account_id').agg(
    total_orders=('order_id', 'nunique'),
    total_spend=('price', 'sum'),
    last_order_date=('order_date', 'max'),
    avg_review_score=('review_score', 'mean'),
    late_deliveries=('is_late', lambda x: (x == True).sum()),
    distinct_categories=('product_category', 'nunique'),
    avg_freight_ratio=('freight_ratio', 'mean')
).reset_index()

# Calculate Recency and clean up
base_features['recency_days'] = (cutoff_date - base_features['last_order_date']).dt.days
base_features['avg_review_score'] = base_features['avg_review_score'].fillna(base_features['avg_review_score'].median())
base_features = base_features.drop(columns=['last_order_date'])

# Merge with Payments Data to capture Financial Friction
valid_orders = feature_data[['order_id', 'account_id']].drop_duplicates()
pay_merged = pd.merge(valid_orders, df_payments, on='order_id', how='inner')

pay_features = pay_merged.groupby('account_id').agg(
    boleto_count=('payment_type', lambda x: (x == 'boleto').sum()),
    total_payments=('payment_type', 'count'),
    avg_installments=('payment_installments', 'mean')
).reset_index()

pay_features['pct_boleto'] = pay_features['boleto_count'] / pay_features['total_payments']
pay_features = pay_features[['account_id', 'pct_boleto', 'avg_installments']]

# Combine Base and Payment Features
features = pd.merge(base_features, pay_features, on='account_id', how='left').fillna(0)
print("Base features and payment features engineered.")

Base features and payment features engineered.


BLOCK 5: The Decline Metrics (Recent 90 Days vs Previous 90 Days)

In [5]:
# Define the exact 90-day periods
recent_90_start = cutoff_date - pd.Timedelta(days=90)
prev_90_start = recent_90_start - pd.Timedelta(days=90)

# Filter the raw feature data into the two periods
recent_data = feature_data[(feature_data['order_date'] > recent_90_start) & (feature_data['order_date'] <= cutoff_date)]
prev_data = feature_data[(feature_data['order_date'] > prev_90_start) & (feature_data['order_date'] <= recent_90_start)]

# Aggregate Spend and Orders for Recent 90 Days
recent_agg = recent_data.groupby('account_id').agg(
    recent_spend=('price', 'sum'),
    recent_orders=('order_id', 'nunique')
).reset_index()

# Aggregate Spend and Orders for Previous 90 Days
prev_agg = prev_data.groupby('account_id').agg(
    prev_spend=('price', 'sum'),
    prev_orders=('order_id', 'nunique')
).reset_index()

# Merge trend data into our main features dataset
features = pd.merge(features, recent_agg, on='account_id', how='left').fillna(0)
features = pd.merge(features, prev_agg, on='account_id', how='left').fillna(0)

# Calculate the actual Decline Metrics
# Ratio of recent spend to previous spend (+1 to avoid division by zero)
features['spend_trend_ratio'] = features['recent_spend'] / (features['prev_spend'] + 1)
# Difference in order velocity
features['order_velocity_change'] = features['recent_orders'] - features['prev_orders']

# Drop the raw temporary columns to keep the dataset clean
features = features.drop(columns=['recent_spend', 'recent_orders', 'prev_spend', 'prev_orders'])
print("Decline metrics (spend_trend_ratio & order_velocity_change) calculated.")

Decline metrics (spend_trend_ratio & order_velocity_change) calculated.


BLOCK 6: Defining Churn Target & Model Training

In [6]:
# 1. Define Churn (Target Window: July 1 - Aug 31)
# Anyone who placed an order in the target data is active. The rest have churned.
active_accounts = target_data['account_id'].unique()
features['churned'] = np.where(features['account_id'].isin(active_accounts), 0, 1)

print(f"Total established accounts evaluated: {len(features)}")
print(f"Accounts labeled as Churned (0 orders in last 60 days): {features['churned'].sum()}")

# 2. Train the Model to generate Evidence
X = features.drop(columns=['account_id', 'churned'])
y = features['churned']

# Train Random Forest with class weights to handle the churn imbalance
rf = RandomForestClassifier(n_estimators=150, class_weight='balanced', random_state=42)
rf.fit(X, y)

# 3. Output the Feature Importance Evidence
evidence_df = pd.DataFrame({
    'Feature': X.columns, 
    'Importance_Weight': rf.feature_importances_
}).sort_values('Importance_Weight', ascending=False)

print("\n--- ALGORITHMIC EVIDENCE FOR CHURN DRIVERS ---")
print(evidence_df.to_string(index=False))

Total established accounts evaluated: 2149
Accounts labeled as Churned (0 orders in last 60 days): 501

--- ALGORITHMIC EVIDENCE FOR CHURN DRIVERS ---
              Feature  Importance_Weight
         recency_days           0.145774
    spend_trend_ratio           0.143677
          total_spend           0.109423
    avg_freight_ratio           0.093236
     avg_installments           0.092610
     avg_review_score           0.089344
         total_orders           0.085962
           pct_boleto           0.079715
  distinct_categories           0.070549
order_velocity_change           0.055297
      late_deliveries           0.034413


1. Defining the Business Entity (The Account):Instead of looking at individual buyers, the data was grouped by zip code area (account_id), as 97% of individual buyers only order once. This matches the requirement to treat each zip code as a distinct B2B account (like a bar or shop).   2. Filtering the Target Audience (Removing Noise):The algorithm was restricted to evaluate only accounts with 10 or more historical orders.This criterion was chosen because these roughly 2,600 accounts drive 52% of all volume and order in a median of 10 different months. Filtering out the smaller accounts ensures the model learns to identify real business decline rather than getting confused by the random order gaps typical of one-off buyers.   3. Establishing the Time Windows (Preventing Data Leakage):Because the dataset ends on August 31, 2018, this date was treated as "today".   The Target Window: July 1, 2018, to August 31, 2018. This 60-day period was used strictly to check the "answer" (did they churn or not?).The Feature Window: January 2017 to June 30, 2018. The algorithm was only allowed to calculate behavioral signals using data before July 1. This prevents the model from cheating by looking at the outcome it is supposed to predict.4. Defining the Churn Target (The Variable to Predict):If an established account placed zero orders during the Target Window, they were mathematically labeled as "Churned" (1). If they placed at least one order, they were labeled as "Active" (0).This defines churn as an account skipping months or dropping off entirely.   5. Engineering the Input Criteria (The Signals):Using the Feature Window data, the algorithm calculated baseline rhythms (total spend, total orders, distinct categories) and experience metrics (review scores, late deliveries) to see if a bad experience caused them to leave.   To account for the fact that order volume grew strongly during 2017, "decline criteria" were engineered. The algorithm specifically calculated the ratio of spend and order velocity in the most recent 90 days compared to the previous 90 days to identify accounts that were actively shrinking prior to the cutoff.   6. Algorithmic Extraction (How the Weights are Assigned):Both the Active and Churned accounts were fed into a Random Forest Classifier.Because there are fewer churned accounts than active ones, the algorithm used "balanced class weights" to penalize itself heavily if it failed to identify the traits of a leaving customer.The model built hundreds of decision trees to try and separate the Active accounts from the Churned accounts. The "Importance Weights" it outputted are a mathematical measure of how often a specific feature (like recency_days) was successfully used by the trees to accurately split a churner from a non-churner.

BLOCK 7: Risk Score & Prioritization

In [ ]:
# 1. Get the raw probability of churn for each account (Probability of Class 1)
# predict_proba returns an array like [prob_active, prob_churned]
features['churn_probability'] = rf.predict_proba(X)[:, 1]

# 2. Calculate the Risk Score (Probability * Total Spend)
# This fulfills the business requirement to prioritize based on value
features['risk_score'] = features['churn_probability'] * features['total_spend']

# 3. Create the final prioritized action list
# We only want to flag accounts that are actually active in our feature window,
# so we look at accounts with a risk score > 0, sorted highest to lowest.
action_list = features[['account_id', 'churn_probability', 'total_spend', 'risk_score']].copy()
action_list = action_list.sort_values(by='risk_score', ascending=False)

# 4. Display the Top 10 accounts the sales rep needs to contact today
print("--- TOP 10 ACCOUNTS REQUIRING IMMEDIATE ACTION ---")
print(action_list.head(10).to_string(index=False))

action_list.to_csv('prioritized_at_risk_accounts.csv', index=False)

--- TOP 10 ACCOUNTS REQUIRING IMMEDIATE ACTION ---
account_id  churn_probability  total_spend  risk_score
    A78850           0.900000      6232.81 5609.529000
    A32280           0.913333      6118.74 5588.449200
    A24020           0.886667      6077.03 5388.299933
    A89190           0.893333      5965.23 5328.938800
    A88058           0.906667      5855.71 5309.177067
    A07600           0.926667      5657.94 5243.024400
    A20540           0.966667      5131.80 4960.740000
    A87050           0.913333      5415.72 4946.357600
    A35588           0.906667      5391.73 4888.501867
    A37550           0.933333      5091.31 4751.889333


BLOCK 8: Generating the "Today" Risk List (Deployment)

In [9]:
# BLOCK 8: Generating the "Today" Risk List (Deployment)

# 1. Shift our "cutoff" to today (August 31, 2018)
today_date = pd.to_datetime('2018-08-31')

# 2. Recalculate features using ALL data up to today
current_data = df_lines[df_lines['order_date'] <= today_date].copy()
current_data['freight_ratio'] = current_data['freight_value'] / (current_data['price'] + 0.01)

# Base Features as of Today
current_base = current_data.groupby('account_id').agg(
    total_orders=('order_id', 'nunique'),
    total_spend=('price', 'sum'),
    last_order_date=('order_date', 'max'),
    avg_review_score=('review_score', 'mean'),
    late_deliveries=('is_late', lambda x: (x == True).sum()),
    distinct_categories=('product_category', 'nunique'),
    avg_freight_ratio=('freight_ratio', 'mean')
).reset_index()

# Filter for accounts that are established as of today
current_base = current_base[current_base['total_orders'] >= 10].copy()
current_base['recency_days'] = (today_date - current_base['last_order_date']).dt.days
current_base['avg_review_score'] = current_base['avg_review_score'].fillna(current_base['avg_review_score'].median())
current_base = current_base.drop(columns=['last_order_date'])

# Payment Features as of Today
valid_orders_today = current_data[['order_id', 'account_id']].drop_duplicates()
pay_merged_today = pd.merge(valid_orders_today, df_payments, on='order_id', how='inner')

current_pay = pay_merged_today.groupby('account_id').agg(
    boleto_count=('payment_type', lambda x: (x == 'boleto').sum()),
    total_payments=('payment_type', 'count'),
    avg_installments=('payment_installments', 'mean')
).reset_index()

current_pay['pct_boleto'] = current_pay['boleto_count'] / current_pay['total_payments']
current_pay = current_pay[['account_id', 'pct_boleto', 'avg_installments']]

# Combine
features_today = pd.merge(current_base, current_pay, on='account_id', how='left').fillna(0)

# Decline Metrics (Recent 90 vs Prev 90 as of TODAY)
recent_90_start_today = today_date - pd.Timedelta(days=90)
prev_90_start_today = recent_90_start_today - pd.Timedelta(days=90)

recent_data_today = current_data[(current_data['order_date'] > recent_90_start_today)]
prev_data_today = current_data[(current_data['order_date'] > prev_90_start_today) & (current_data['order_date'] <= recent_90_start_today)]

recent_agg_today = recent_data_today.groupby('account_id').agg(recent_spend=('price', 'sum'), recent_orders=('order_id', 'nunique')).reset_index()
prev_agg_today = prev_data_today.groupby('account_id').agg(prev_spend=('price', 'sum'), prev_orders=('order_id', 'nunique')).reset_index()

features_today = pd.merge(features_today, recent_agg_today, on='account_id', how='left').fillna(0)
features_today = pd.merge(features_today, prev_agg_today, on='account_id', how='left').fillna(0)

features_today['spend_trend_ratio'] = features_today['recent_spend'] / (features_today['prev_spend'] + 1)
features_today['order_velocity_change'] = features_today['recent_orders'] - features_today['prev_orders']
features_today = features_today.drop(columns=['recent_spend', 'recent_orders', 'prev_spend', 'prev_orders'])

# 3. Predict the FUTURE using the trained model
X_today = features_today.drop(columns=['account_id'])

# Make sure columns are in the exact same order as training
X_today = X_today[X.columns]

# Predict probability of churn for Sept/Oct
features_today['churn_probability'] = rf.predict_proba(X_today)[:, 1]

# 4. Calculate Final Risk Score for TODAY
features_today['risk_score'] = features_today['churn_probability'] * features_today['total_spend']

final_action_list = features_today[['account_id', 'churn_probability', 'total_spend', 'risk_score']].sort_values(by='risk_score', ascending=False)

print("--- CURRENT TOP 10 AT-RISK ACCOUNTS (AS OF AUG 31) ---")
print(final_action_list.head(10).to_string(index=False))

# Save this true list for your prototype!
final_action_list.head(50).to_csv('current_at_risk_accounts.csv', index=False)

--- CURRENT TOP 10 AT-RISK ACCOUNTS (AS OF AUG 31) ---
account_id  churn_probability  total_spend  risk_score
    A35501           0.326667     18061.03 5899.936467
    A07600           0.853333      5657.94 4828.108800
    A37550           0.933333      5091.31 4751.889333
    A20540           0.906667      5131.80 4652.832000
    A13348           0.886667      4700.47 4167.750067
    A22775           0.980000      4177.31 4093.763800
    A50050           0.793333      4815.00 3819.900000
    A24020           0.620000      6077.03 3767.758600
    A96211           0.953333      3838.15 3659.036333
    A22290           0.786667      4421.81 3478.490533
